# Risk fusion demo — run the final model on your own photos

Runs the complete pipeline on any image you upload: child detector at
imgsz 416, hazard detector at 640, then the distance rule that produces the
**Safe / Unsafe** label.

Unlike the training notebooks, this one needs **no GPU, no Roboflow key and
no dataset download**. The two weight files are committed to the repo (12 MB),
so cloning is enough. A CPU runtime is fine — about 6 s of startup, then
~30 ms per frame.

Defaults are the validation-calibrated ones from Phase 3c: centroid distance,
threshold 0.3625, detector confidence 0.05. This notebook computes no metrics
and reads no ground truth, so nothing here can touch a held-out split.

In [ ]:
!pip install -q ultralytics==8.4.106

In [ ]:
import os
REPO_URL = "https://github.com/FooJames/DEEPLRN_Group2.git"
if not os.path.isdir("DEEPLRN_Group2"):
    !git clone $REPO_URL
else:
    !cd DEEPLRN_Group2 && git pull
%cd DEEPLRN_Group2
!ls -la models/

## Upload some images

Pick anything with a child and a household hazard in frame. The repo ships no
sample photos — `data/` is gitignored, so the co-occurrence evaluation images
are not in the clone. Rebuilding those needs the Roboflow key and
`make_cooccurrence_eval.py`; for a demo, your own pictures are the point.

In [ ]:
import os, shutil
from google.colab import files

os.makedirs("demo_in", exist_ok=True)
for name in files.upload():
    shutil.move(name, os.path.join("demo_in", name))
print(sorted(os.listdir("demo_in")))

## Run the pipeline

One line per image: the verdict, the normalised min child-hazard distance,
and how many boxes each detector returned. `d = n/a` means one of the
detectors found nothing, which the fusion rule scores Safe by definition —
that is a detection failure, not a safety judgement.

In [ ]:
!python scripts/risk_fusion.py demo_in --out runs/risk_fusion

In [ ]:
import glob
from IPython.display import Image, display

for p in sorted(glob.glob("runs/risk_fusion/*_risk.jpg")):
    print(p)
    display(Image(p, width=720))

## Why the confidence is set so low

The run above used `--conf 0.05`. Re-run it at ultralytics' default 0.25 and
compare the hazard counts: at 0.25 the hazard detector misses about 57% of
hazards, and every miss is silently scored Safe. The low threshold buys recall
at the cost of spurious hazard boxes in cluttered frames.

Watch for frames that flip from **UNSAFE** to `d = n/a`. That is not the model
judging the scene safe — it is the hazard detector finding nothing and the
fusion rule defaulting. On six sample evaluation images, raising the confidence
to 0.25 flipped one genuinely unsafe frame (d = 0.1028) to Safe and took the
no-detection count from 1 to 3.

This is the pipeline's real bottleneck — detection dominates geometry (§8 of
`results_and_findings.md`), which is also why centroid-vs-edge came out a null
result.

In [ ]:
!python scripts/risk_fusion.py demo_in --conf 0.25 --no-save

## Take the annotated frames with you

In [ ]:
!zip -qr risk_fusion_demo.zip runs/risk_fusion
!unzip -l risk_fusion_demo.zip | tail -4
from google.colab import files
files.download("risk_fusion_demo.zip")

### Notes
- `--metric edge` swaps centroid distance for nearest-edge distance. The
  threshold 0.3625 was calibrated for **centroid**; edge needs its own
  (see `results/metrics/ablation_distance_ref_val.csv`).
- The two detectors run at different resolutions on purpose. Don't pass a
  single `imgsz` for both.
- Numbers for the paper come from `results/metrics/`, not from this notebook.
- If the clone asks for credentials, the repo is private — use a personal
  access token in the URL, or upload `models/` and `scripts/` by hand.